In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

df = pd.read_csv("../data/processed/strixhaven_clean.csv")
print(df.shape)

In [ ]:
train, test = train_test_split(df, test_size=0.2, random_state=42)
print(f"Train: {len(train)} cards, Test: {len(test)} cards")

In [ ]:
mean_win_rate = train["win_rate"].mean()
print(f"Training set mean win rate: {mean_win_rate:.4f}")

test_predictions_mean = np.full(len(test), mean_win_rate)

rmse_mean = mean_squared_error(test["win_rate"], test_predictions_mean) ** 0.5
r2_mean = r2_score(test["win_rate"], test_predictions_mean)

print(f"Mean baseline  ->  RMSE: {rmse_mean:.4f}   R²: {r2_mean:.4f}")

In [ ]:
train = train.copy()
test = test.copy()
train["text_length"] = train["text"].astype(str).str.len()
test["text_length"] = test["text"].astype(str).str.len()

model_length = LinearRegression()
model_length.fit(train[["text_length"]], train["win_rate"])

test_predictions_length = model_length.predict(test[["text_length"]])

rmse_length = mean_squared_error(test["win_rate"], test_predictions_length) ** 0.5
r2_length = r2_score(test["win_rate"], test_predictions_length)

print(f"Text length regression  ->  RMSE: {rmse_length:.4f}   R²: {r2_length:.4f}")

In [ ]:
def parse_mana_value(cost):
    """Convert a mana cost string like '{4}{W}' into a total numeric mana value."""
    if not isinstance(cost, str) or cost == "":
        return 0
    tokens = re.findall(r"\{(.*?)\}", cost)
    total = 0
    for t in tokens:
        if t.upper() == "X":
            continue  # X costs vary by situation; treat as 0 for this simple baseline
        try:
            total += int(t)       # numeric symbols like {4}
        except ValueError:
            total += 1            # colored symbols like {W}, {U}, hybrid symbols, etc.
    return total

train["mana_value"] = train["manaCost"].apply(parse_mana_value)
test["mana_value"] = test["manaCost"].apply(parse_mana_value)

model_mana = LinearRegression()
model_mana.fit(train[["mana_value"]], train["win_rate"])

test_predictions_mana = model_mana.predict(test[["mana_value"]])

rmse_mana = mean_squared_error(test["win_rate"], test_predictions_mana) ** 0.5
r2_mana = r2_score(test["win_rate"], test_predictions_mana)

print(f"Mana value regression  ->  RMSE: {rmse_mana:.4f}   R²: {r2_mana:.4f}")

In [ ]:
results = pd.DataFrame({
    "Model": ["Mean baseline", "Text length only", "Mana value only"],
    "RMSE": [rmse_mean, rmse_length, rmse_mana],
    "R²": [r2_mean, r2_length, r2_mana],
})
print(results.to_string(index=False))